# 02 · Azure OpenAI GPT-5.1 Baseline Evaluation

Runs the test dataset through **GPT-5.1** via Azure OpenAI and score each generated query with the same three-component reward used during GRPO
training (format · execution · schema fidelity). This will be used as a baseline to compare with Instruction fine-tuned and GRPO fine tuned SLM models


In [7]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

# Load variables from .env (copy .env.param → .env and fill in your values)
_env_file = REPO_ROOT / ".env"
if _env_file.exists():
    load_dotenv(_env_file, override=True)
    print(f"Loaded environment from {_env_file}")
else:
    print(f"WARNING: {_env_file} not found. Copy .env.param → .env and fill in your values.")

# ── Azure OpenAI ──────────────────────────────────────────────────────────
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT", "https://<your-resource>.openai.azure.com/")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5.1")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
TEMPERATURE              = float(os.getenv("TEMPERATURE", "0.0"))
MAX_TOKENS               = int(os.getenv("MAX_TOKENS", "512"))

# ── Data ──────────────────────────────────────────────────────────────────
CSV_PATH    = REPO_ROOT / "data" / "test_dataset_baseline.csv"

# Resolve RAWDATA_DIR to an absolute path anchored to REPO_ROOT (not Jupyter's CWD).
_rawdata = Path(os.getenv("RAWDATA_DIR", "data"))
if not _rawdata.is_absolute():
    _rawdata = REPO_ROOT / _rawdata
RAWDATA_DIR = str(_rawdata.resolve())
os.environ["RAWDATA_DIR"] = RAWDATA_DIR  # ensure submodules pick up the absolute path

if not AZURE_OPENAI_API_KEY:
    print("WARNING: AZURE_OPENAI_API_KEY is not set.")

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"CSV_PATH    : {CSV_PATH}")
print(f"RAWDATA_DIR : {RAWDATA_DIR}")
print(f"ENDPOINT    : {AZURE_OPENAI_ENDPOINT}")
print(f"DEPLOYMENT  : {AZURE_OPENAI_DEPLOYMENT}")


Loaded environment from C:\code\text2sql-slm-finetuning-grpo\.env
REPO_ROOT   : C:\code\text2sql-slm-finetuning-grpo
CSV_PATH    : C:\code\text2sql-slm-finetuning-grpo\data\test_dataset_baseline.csv
RAWDATA_DIR : C:\code\text2sql-slm-finetuning-grpo\data
ENDPOINT    : https://vism-mjcnt3i9-eastus2.openai.azure.com/
DEPLOYMENT  : gpt-5.2-chat


## 1. Setup Azure OpenAI Client


In [8]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("Azure OpenAI client ready.")

response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[{"role": "user", "content": "Tell me a joke."}],
        max_completion_tokens=MAX_TOKENS,
)

response.choices[0].message.content


Azure OpenAI client ready.


'Why don’t scientists trust atoms?\n\nBecause they make up everything! 😄'

## 2. Load Test Dataset


In [9]:
import ast
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
print(f"Sources : {df['source'].value_counts().to_dict()}")
df.head(1)

Loaded 240 rows
Sources : {'bird': 162, 'spider': 78}


,prompt,solution,schema,source,db_id
0,"[{'role': 'system', 'content': 'You are an exp...",SELECT T2.major_name FROM member AS T1 INNER J...,"{'event': ['event_id', 'event_name', 'event_da...",bird,student_club


## 3. Run Azure OpenAI Inference


In [10]:
import ast
from tqdm.auto import tqdm


def call_azure_openai(messages: list[dict]) -> tuple[str, int, int]:
    """Send a chat-format prompt to Azure OpenAI and return (text, prompt_tokens, completion_tokens)."""
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        max_completion_tokens=MAX_TOKENS,
    )
    text = response.choices[0].message.content or ""
    usage = response.usage
    return text, (usage.prompt_tokens if usage else 0), (usage.completion_tokens if usage else 0)


completions, prompt_tokens_list, completion_tokens_list = [], [], []
for raw_prompt in tqdm(df["prompt"], desc=f"Inference ({AZURE_OPENAI_DEPLOYMENT})"):
    try:
        messages = ast.literal_eval(raw_prompt)
        text, pt, ct = call_azure_openai(messages)
    except Exception as exc:
        print(f"API error: {exc}")
        text, pt, ct = "", 0, 0
    completions.append(text)
    prompt_tokens_list.append(pt)
    completion_tokens_list.append(ct)

df["gpt_completion"]       = completions
df["prompt_tokens"]        = prompt_tokens_list
df["completion_tokens"]    = completion_tokens_list
df["total_tokens"]         = df["prompt_tokens"] + df["completion_tokens"]

print(f"\n{sum(bool(c) for c in completions)}/{len(completions)} non-empty responses.")
print(f"Total tokens consumed — prompt: {df['prompt_tokens'].sum():,}  completion: {df['completion_tokens'].sum():,}  total: {df['total_tokens'].sum():,}")
df[["db_id", "gpt_completion", "prompt_tokens", "completion_tokens", "total_tokens"]].head(3)


Inference (gpt-5.2-chat): 100%|██████████| 240/240 [21:13<00:00,  5.31s/it]


240/240 non-empty responses.
Total tokens consumed — prompt: 73,727  completion: 32,366  total: 106,093


,db_id,gpt_completion,prompt_tokens,completion_tokens,total_tokens
0,student_club,```sql\nSELECT m.major_name\nFROM member AS me...,266,58,324
1,student_club,```sql\nSELECT COUNT(*) AS engineering_student...,275,114,389
2,student_club,"```sql\nSELECT CONCAT(m.first_name, ' ', m.las...",283,185,468


## 4. Compute Rewards


In [11]:
import ast
from rewards import combined_reward
from loguru import logger
import sys
logger.remove()
logger.add(sys.stdout, level="DEBUG")

# Build TRL-format inputs ─ completions as list[list[dict]]
trl_completions = [[{"role": "assistant", "content": c}] for c in df["gpt_completion"]]
trl_prompts     = df["prompt"].apply(ast.literal_eval).tolist()
schemas         = df["schema"].apply(ast.literal_eval).tolist()
sources         = df["source"].tolist()
db_ids          = df["db_id"].tolist()

scores = combined_reward(
    completions=trl_completions,
    prompts=trl_prompts,
    schemas=schemas,
    db_paths=db_ids,
    source=sources,
)

df["gpt_reward"] = scores
print(df["gpt_reward"].describe().to_string())


2026-03-10 19:07:06.889 | DEBUG    | rewards:_exec_on_sqlite:180 - [exec] Resolved full_path='C:\\code\\text2sql-slm-finetuning-grpo\\data/bird/dev_20240627/dev_databases/student_club/student_club.sqlite'
2026-03-10 19:07:06.897 | DEBUG    | rewards:_exec_on_sqlite:180 - [exec] Resolved full_path='C:\\code\\text2sql-slm-finetuning-grpo\\data/bird/dev_20240627/dev_databases/student_club/student_club.sqlite'
2026-03-10 19:07:06.901 | DEBUG    | rewards:_exec_on_sqlite:180 - [exec] Resolved full_path='C:\\code\\text2sql-slm-finetuning-grpo\\data/bird/dev_20240627/dev_databases/student_club/student_club.sqlite'
2026-03-10 19:07:06.902 | DEBUG    | rewards:_exec_on_sqlite:180 - [exec] Resolved full_path='C:\\code\\text2sql-slm-finetuning-grpo\\data/bird/dev_20240627/dev_databases/student_club/student_club.sqlite'
2026-03-10 19:07:06.908 | DEBUG    | rewards:_exec_on_sqlite:180 - [exec] Resolved full_path='C:\\code\\text2sql-slm-finetuning-grpo\\data/bird/dev_20240627/dev_databases/student_c

## 5. Average Reward


In [12]:
overall_avg = df["gpt_reward"].mean()
per_source  = df.groupby("source")["gpt_reward"].mean()

print(f"── GPT-5.1 Reward Summary ──────────────────────────────")
print(f"  Overall average : {overall_avg:.4f}")
print()
print("  Per-source average:")
for src, val in per_source.items():
    print(f"    {src:10s}: {val:.4f}")


── GPT-5.1 Reward Summary ──────────────────────────────
  Overall average : 0.9801

  Per-source average:
    bird      : 0.9770
    spider    : 0.9865


In [ ]:
print(f"── Token Usage Summary ─────────────────────────────────")
print(f"  Prompt tokens     : {df['prompt_tokens'].sum():>10,}")
print(f"  Completion tokens : {df['completion_tokens'].sum():>10,}")
print(f"  Total tokens      : {df['total_tokens'].sum():>10,}")
print()
print("  Per-source token breakdown:")
print(df.groupby("source")[["prompt_tokens", "completion_tokens", "total_tokens"]].sum().to_string())

── Token Usage Summary ─────────────────────────────────
  Prompt tokens     :     73,727
  Completion tokens :     32,366
  Total tokens      :    106,093

  Per-source token breakdown:
        prompt_tokens  completion_tokens  total_tokens
source                                                
bird            44841              23143         67984
spider          28886               9223         38109
